In [ ]:
import folium
from folium import PolyLine, CircleMarker
import json
import numpy as np


# -------------------------------------------------------------
# Convert meters → kilometers
# -------------------------------------------------------------
def km(err_meters):
    return round(err_meters / 1000, 1) if err_meters is not None else None


# -------------------------------------------------------------
# Extract [lat, lon] from [lat, lon, t]
# -------------------------------------------------------------
def xy(list3):
    return [[p[0], p[1]] for p in list3]


# -------------------------------------------------------------
# Add a LINE + POINTS with time-tooltip
# -------------------------------------------------------------
def draw_line_with_points(map_obj, points3, color, label):
    """
    points3 = [[lat, lon, t], ...]
    color   = line and point color
    label   = name appearing in tooltip/legend
    """

    if len(points3) < 1:
        return

    # Draw the line
    coords = [[p[0], p[1]] for p in points3]
    PolyLine(coords, color=color, weight=4, opacity=0.9, tooltip=label).add_to(map_obj)

    # Draw point markers with tooltip "t = X min"
    for p in points3:
        lat, lon, t = p
        CircleMarker(
            location=[lat, lon],
            radius=4,
            color=color,
            fill=True,
            fill_color=color,
            tooltip=f"{label} — t = {round(t, 2)} min"
        ).add_to(map_obj)


# -------------------------------------------------------------
# MAIN FUNCTION: Create folium map for one flight phase
# -------------------------------------------------------------
def plot_folium_for_phase(phase, rag_data, lstm_data, kf_data, outfile=None):

    print(f"🌍 Creating Folium map for phase: {phase}")

    rag  = rag_data[phase]
    lstm = lstm_data[phase]
    kf = kf_data[phase]
    # -------- Extract triplets (lat, lon, time) --------
    observed      = rag["observed"]
    ground_truth  = rag["ground_truth"]
    rag_pred      = rag["predicted"]
    lstm_pred     = lstm["predicted"]
    kf_pred       = [p[:3] for p in kf["predicted"]]      # KF returns 4 values → keep first 3
    mean_sim      = rag.get("mean_similar_flight", [])

    # -------- Convert errors to kilometers --------
    rag_err  = km(rag["mean_haversine_error"])
    lstm_err = km(lstm["mean_haversine_error"])
    kf_err   = km(kf["mean_haversine_error"])

    # -------- Center map on first observed point --------
    center_lat, center_lon = observed[0][0], observed[0][1]
    m = folium.Map(location=[center_lat, center_lon], zoom_start=7)

    # -----------------------------------------------------
    # DRAW ALL TRAJECTORIES WITH TOOLTIP POINTS
    # -----------------------------------------------------

    # Observed (before spoofing)
    draw_line_with_points(m, observed, "green", "Observed")

    # Ground truth (after spoofing)
    draw_line_with_points(m, ground_truth, "darkgreen", "Ground truth")

    # RAG predictions
    draw_line_with_points(m, rag_pred, "red", f"RAG ({rag_err} km)")

    # LSTM predictions
    draw_line_with_points(m, lstm_pred, "blue", f"LSTM ({lstm_err} km)")

    # Kalman predictions
    draw_line_with_points(m, kf_pred, "orange", f"Kalman ({kf_err} km)")

    # Mean similar flights
    draw_line_with_points(m, mean_sim, "purple", "Mean similar flight")

    # -----------------------------------------------------
    # LEGEND (custom HTML)
    # -----------------------------------------------------
    legend_html = f"""
    <div style="
        position: fixed; 
        bottom: 20px; left: 20px; width: 240px; 
        background-color: white; z-index:9999; 
        padding: 10px; border: 2px solid grey; font-size:14px;">
        <b>Legend — {phase}</b><br>
        <span style="color:green;">■</span> Observed<br>
        <span style="color:darkgreen;">■</span> Ground truth<br>
        <span style="color:red;">■</span> RAG ({rag_err} km)<br>
        <span style="color:blue;">■</span> LSTM ({lstm_err} km)<br>
        <span style="color:orange;">■</span> Kalman ({kf_err} km)<br>
        <span style="color:purple;">■</span> Mean similar flight<br>
        <hr>
        <i>Hover on points to see t (minutes)</i>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))

    # -----------------------------------------------------
    # SAVE FILE
    # -----------------------------------------------------
    if outfile is None:
        outfile = f"map_{phase}.html"

    m.save(outfile)
    print(f"📁 Saved map: {outfile}\n")

    return m


In [ ]:
with open("data/CDG-FCO/results_RAG.json", "r") as f:
    rag_data = json.load(f)

with open("data/CDG-FCO/results_LSTM.json", "r") as f:
    lstm_data = json.load(f)

with open("data/CDG-FCO/results_KALMAN.json", "r") as f:
    kf_data = json.load(f)

with open("data/CDG-FCO/results_BILSTM.json", "r") as f:
    bilstm_data = json.load(f)

m_take_off = plot_folium_for_phase("take_off", rag_data, lstm_data, kf_data)
m_take_off.save("map_take_off.html")
m_cruising = plot_folium_for_phase("cruising", rag_data, lstm_data, kf_data)
m_cruising.save("map_cruising_presenation_boeing.html")
m_landing = plot_folium_for_phase("landing", rag_data, lstm_data, kf_data)
m_landing.save("landing.html")


In [ ]:
import folium
from folium import PolyLine, CircleMarker


In [ ]:
def meters_to_km(err_m):
    return round(err_m / 1000, 1) if err_m is not None else None

In [ ]:
def draw_trajectory(layer, points, color, label):
    """
    points = [[lat, lon, t], ...]
    """

    if not points:
        return

    coords = [[p[0], p[1]] for p in points]

    PolyLine(
        coords,
        color=color,
        weight=4,
        opacity=0.9,
        tooltip=label
    ).add_to(layer)

    for lat, lon, t in points:
        CircleMarker(
            location=[lat, lon],
            radius=4,
            color=color,
            fill=True,
            fill_color=color,
            tooltip=f"{label} — t = {round(t, 2)} min"
        ).add_to(layer)


In [ ]:
def plot_folium_for_phase(
    phase,
    rag_data,
    lstm_data,
    kalman_data,
    outfile=None
):
    print(f"🌍 Creating interactive map for phase: {phase}")

    # ---------------------------
    # Extract data
    # ---------------------------
    rag   = rag_data[phase]
    lstm  = lstm_data[phase]
    kf    = kalman_data[phase]

    observed     = rag["observed"]
    ground_truth = rag["ground_truth"]

    rag_pred  = rag["predicted"]
    lstm_pred = lstm["predicted"]
    kf_pred   = [p[:3] for p in kf["predicted"]]

    mean_sim  = rag.get("mean_similar_flight", [])

    # ---------------------------
    # Errors (km)
    # ---------------------------
    rag_err  = meters_to_km(rag["mean_haversine_error"])
    lstm_err = meters_to_km(lstm["mean_haversine_error"])
    kf_err   = meters_to_km(kf["mean_haversine_error"])

    # ---------------------------
    # Initialize map
    # ---------------------------
    start_lat, start_lon = observed[0][0], observed[0][1]
    m = folium.Map(location=[start_lat, start_lon], zoom_start=7)

    # =====================================================
    # BASE LAYERS (VISIBLE BY DEFAULT)
    # =====================================================
    observed_layer = folium.FeatureGroup(
        name="Observed",
        show=True
    )

    gt_layer = folium.FeatureGroup(
        name="Ground truth",
        show=True
    )

    draw_trajectory(observed_layer, observed, "green", "Observed")
    draw_trajectory(gt_layer, ground_truth, "darkgreen", "Ground truth")

    observed_layer.add_to(m)
    gt_layer.add_to(m)

    # =====================================================
    # MODEL LAYERS (HIDDEN BY DEFAULT)
    # =====================================================
    rag_layer = folium.FeatureGroup(
        name=f"RAG ({rag_err} km)",
        show=False
    )

    lstm_layer = folium.FeatureGroup(
        name=f"LSTM ({lstm_err} km)",
        show=False
    )

    kalman_layer = folium.FeatureGroup(
        name=f"Kalman ({kf_err} km)",
        show=False
    )

    mean_layer = folium.FeatureGroup(
        name="Mean similar flight",
        show=False
    )

    draw_trajectory(rag_layer, rag_pred, "red", "RAG")
    draw_trajectory(lstm_layer, lstm_pred, "blue", "LSTM")
    draw_trajectory(kalman_layer, kf_pred, "orange", "Kalman")
    draw_trajectory(mean_layer, mean_sim, "purple", "Mean similar flight")

    rag_layer.add_to(m)
    lstm_layer.add_to(m)
    kalman_layer.add_to(m)
    mean_layer.add_to(m)

    # =====================================================
    # INTERACTIVE LEGEND
    # =====================================================
    folium.LayerControl(collapsed=False).add_to(m)

    # # ---------------------------
    # # Save map
    # # ---------------------------
    # if outfile is None:
    #     outfile = f"map_{phase}.html"

    # m.save(outfile)
    # print(f"📁 Saved: {outfile}")

    return m


In [ ]:
with open("data/CDG-FCO/results_RAG.json", "r") as f:
    rag_data = json.load(f)

with open("data/CDG-FCO/results_LSTM.json", "r") as f:
    lstm_data = json.load(f)

with open("data/CDG-FCO/results_KALMAN.json", "r") as f:
    kf_data = json.load(f)

with open("data/CDG-FCO/results_BILSTM.json", "r") as f:
    bilstm_data = json.load(f)

m_take_off = plot_folium_for_phase("take_off", rag_data, lstm_data, kf_data)
m_take_off.save("map_take_off.html")
m_cruising = plot_folium_for_phase("cruising", rag_data, lstm_data, kf_data)
m_cruising.save("map_cruising_presenation_boeing.html")
m_landing = plot_folium_for_phase("landing", rag_data, lstm_data, kf_data)
m_landing.save("landing.html")
